# Chapter 4.8: LoRA Serving — Multi-LoRA Architecture in vLLM

This notebook explores how vLLM enables serving multiple LoRA (Low-Rank Adaptation) adapters simultaneously on top of a single base model. We cover LoRA theory, the multi-LoRA challenge, vLLM's LoRA implementation, SGMV kernels, dynamic loading, and memory management.

## Learning Objectives
1. Understand LoRA theory: low-rank decomposition A*B
2. Grasp the multi-LoRA serving challenge and solutions
3. Implement a simplified LoRA layer from scratch
4. Understand vLLM's LoRAManager and adapter lifecycle
5. Build an LRU-based adapter manager

## Prerequisites
- Linear algebra basics (matrix multiplication, rank)
- PyTorch module system
- Basic understanding of transformer architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part4/chapter_4.8_lora.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part4/chapter_4.8_lora.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Optional, Tuple
from collections import OrderedDict
import time
import json
from dataclasses import dataclass, field

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

---
## Part 1: LoRA Theory — Low-Rank Adaptation

### The Key Insight

Fine-tuning a large model updates a weight matrix `W` by adding a delta: `W' = W + ΔW`. The LoRA paper (Hu et al., 2021) showed that this `ΔW` has low intrinsic rank, so it can be decomposed:

```
ΔW = B × A

where:
  W is (d_out × d_in)     — original weight, e.g., (4096 × 4096)
  A is (r × d_in)         — down-projection, r << d_in (e.g., r=16)
  B is (d_out × r)        — up-projection
  ΔW is (d_out × d_in)    — the LoRA update
```

**Parameters saved**: Instead of `d_out × d_in` (16M for 4096×4096), we only store `r × (d_in + d_out)` (131K for r=16). That's a **~122× reduction**.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Figure: LoRA Architecture Diagram ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')
fig.patch.set_facecolor('white')

# Input x
ax.add_patch(mpatches.FancyBboxPatch((6, 6.8), 2, 0.8, boxstyle="round,pad=0.1",
    facecolor='#95A5A6', alpha=0.8, edgecolor='white', linewidth=2))
ax.text(7, 7.2, 'x\n(d_in)', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Original Weight W (frozen) - left branch
ax.add_patch(mpatches.FancyBboxPatch((2, 4.5), 3, 1.5, boxstyle="round,pad=0.12",
    facecolor=BLUE, alpha=0.9, edgecolor='white', linewidth=2.5))
ax.text(3.5, 5.55, 'W (frozen)', ha='center', fontsize=11, fontweight='bold', color='white')
ax.text(3.5, 4.9, 'd_out x d_in\n(e.g. 4096 x 4096)', ha='center', fontsize=8, color='white')
ax.annotate('', xy=(3.5, 6.0), xytext=(6, 7.0),
            arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

# LoRA branch - right side
# A matrix (down-projection)
ax.add_patch(mpatches.FancyBboxPatch((9, 5.2), 2.5, 1.0, boxstyle="round,pad=0.1",
    facecolor=GREEN, alpha=0.9, edgecolor='white', linewidth=2))
ax.text(10.25, 5.9, 'A (trainable)', ha='center', fontsize=9, fontweight='bold', color='white')
ax.text(10.25, 5.4, 'r x d_in\n(16 x 4096)', ha='center', fontsize=7, color='white')
ax.annotate('', xy=(10.25, 6.2), xytext=(8, 7.0),
            arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

# B matrix (up-projection)
ax.add_patch(mpatches.FancyBboxPatch((9, 3.5), 2.5, 1.0, boxstyle="round,pad=0.1",
    facecolor=GREEN, alpha=0.9, edgecolor='white', linewidth=2))
ax.text(10.25, 4.2, 'B (trainable)', ha='center', fontsize=9, fontweight='bold', color='white')
ax.text(10.25, 3.7, 'd_out x r\n(4096 x 16)', ha='center', fontsize=7, color='white')
ax.annotate('', xy=(10.25, 4.5), xytext=(10.25, 5.2),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2))

# Scaling factor
ax.text(12, 4.0, 'x  alpha/r', fontsize=10, fontweight='bold', color=ORANGE)

# Addition at output
ax.add_patch(plt.Circle((7, 2.5), 0.4, facecolor=ORANGE, edgecolor='white', linewidth=2, alpha=0.9))
ax.text(7, 2.5, '+', ha='center', va='center', fontsize=18, fontweight='bold', color='white')

# Arrows to addition
ax.annotate('', xy=(5.5, 2.5), xytext=(3.5, 4.5),
            arrowprops=dict(arrowstyle='->', color=BLUE, lw=2))
ax.text(3.5, 3.3, 'W @ x', fontsize=9, fontweight='bold', color=BLUE)

ax.annotate('', xy=(8.5, 2.5), xytext=(10.25, 3.5),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2))
ax.text(10.25, 2.8, 'B @ A @ x\n* (alpha/r)', fontsize=8, fontweight='bold', color=GREEN)

# Output
ax.add_patch(mpatches.FancyBboxPatch((6, 0.8), 2, 0.8, boxstyle="round,pad=0.1",
    facecolor=PURPLE, alpha=0.85, edgecolor='white', linewidth=2))
ax.text(7, 1.2, 'y = Wx + BAx * s\n(d_out)', ha='center', va='center', fontsize=9,
        fontweight='bold', color='white')
ax.annotate('', xy=(7, 1.6), xytext=(7, 2.1),
            arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

# Dimension callout
ax.add_patch(mpatches.FancyBboxPatch((0.5, 0.5), 4.5, 1.5, boxstyle="round,pad=0.15",
    facecolor='lightyellow', edgecolor=ORANGE, linewidth=1.5))
ax.text(2.75, 1.5, 'Parameter savings:', fontsize=9, fontweight='bold', color='#333')
ax.text(2.75, 1.0, 'W: 4096x4096 = 16.8M\nA+B: 16x(4096+4096) = 131K\nRatio: 128x fewer!',
        fontsize=8, ha='center', color='#555')

ax.set_title('LoRA Architecture: Frozen Base Weight W + Trainable Low-Rank A*B',
             fontsize=15, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

In [ ]:
# Demonstrate the parameter savings from LoRA decomposition

def lora_param_analysis(d_in: int, d_out: int, ranks: List[int]):
    """Analyze parameter count for different LoRA ranks."""
    full_params = d_in * d_out
    
    print(f"Weight matrix: ({d_out} x {d_in}) = {full_params:,} parameters")
    print(f"\n{'Rank r':<10} {'LoRA Params':<15} {'Ratio':<10} {'Savings':<10}")
    print("-" * 45)
    
    ratios = []
    for r in ranks:
        lora_params = r * (d_in + d_out)  # A: (r, d_in) + B: (d_out, r)
        ratio = lora_params / full_params
        ratios.append(ratio)
        print(f"{r:<10} {lora_params:<15,} {ratio:<10.4f} {1/ratio:<10.0f}x")
    
    return ratios

# Typical transformer dimensions
print("=== Llama-3 8B (d=4096) ===")
ratios_8b = lora_param_analysis(4096, 4096, [1, 4, 8, 16, 32, 64, 128, 256])

print("\n=== Llama-3 70B (d=8192) ===")
ratios_70b = lora_param_analysis(8192, 8192, [1, 4, 8, 16, 32, 64, 128, 256])

In [ ]:
# Visualize parameter savings
ranks = [1, 4, 8, 16, 32, 64, 128, 256]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(ranks)), ratios_8b, color='steelblue', alpha=0.8)
axes[0].set_xticks(range(len(ranks)))
axes[0].set_xticklabels(ranks)
axes[0].set_xlabel('LoRA Rank')
axes[0].set_ylabel('Parameter Ratio (LoRA / Full)')
axes[0].set_title('8B Model: LoRA Parameter Ratio')
axes[0].set_yscale('log')

# Memory comparison for multi-LoRA
num_adapters = range(1, 51)
full_ft_memory = [n * 4096 * 4096 * 32 * 4 / (1024**3) for n in num_adapters]  # 32 layers, float32
lora_r16_memory = [n * 16 * (4096 + 4096) * 32 * 4 / (1024**3) for n in num_adapters]
lora_r64_memory = [n * 64 * (4096 + 4096) * 32 * 4 / (1024**3) for n in num_adapters]

axes[1].plot(num_adapters, full_ft_memory, 'r-', linewidth=2, label='Full fine-tuned copies')
axes[1].plot(num_adapters, lora_r64_memory, 'b--', linewidth=2, label='LoRA r=64')
axes[1].plot(num_adapters, lora_r16_memory, 'g-.', linewidth=2, label='LoRA r=16')
axes[1].set_xlabel('Number of Adapters')
axes[1].set_ylabel('Memory (GB)')
axes[1].set_title('Memory for Multi-Adapter Serving')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Implement a basic LoRA layer from scratch

class LoRALayer(nn.Module):
    """A single LoRA adapter layer.
    
    The forward pass computes:
        output = W @ x + (B @ A) @ x * scaling
               = base_output + lora_output * scaling
    
    where scaling = alpha / r
    """
    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 16,
        alpha: float = 16.0,
        dropout: float = 0.0
    ):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # LoRA matrices
        # A: down-projection (r x d_in) — initialized with Kaiming
        # B: up-projection (d_out x r) — initialized with zeros
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        
        # Initialize A with Kaiming uniform
        nn.init.kaiming_uniform_(self.lora_A, a=np.sqrt(5))
        
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute LoRA delta: B @ A @ x * scaling."""
        # x shape: (batch_size, in_features)
        # A @ x: (rank, batch) -> via x @ A^T -> (batch, rank)
        # B @ (A @ x): (batch, out_features)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return lora_out * self.scaling
    
    def get_merged_delta(self) -> torch.Tensor:
        """Compute the full ΔW = B @ A * scaling for weight merging."""
        return (self.lora_B @ self.lora_A) * self.scaling


# Demo: Create and test a LoRA layer
d_in, d_out, rank = 4096, 4096, 16
lora = LoRALayer(d_in, d_out, rank=rank, alpha=16.0)

print(f"LoRA layer: in={d_in}, out={d_out}, rank={rank}")
print(f"  A shape: {lora.lora_A.shape}")
print(f"  B shape: {lora.lora_B.shape}")
print(f"  Total parameters: {sum(p.numel() for p in lora.parameters()):,}")
print(f"  Scaling factor: {lora.scaling}")

# Forward pass
x = torch.randn(4, d_in)  # batch of 4
delta = lora(x)
print(f"\n  Input shape: {x.shape}")
print(f"  Output (delta) shape: {delta.shape}")
print(f"  Output norm: {delta.norm().item():.4f}")

In [ ]:
# LoRA-augmented linear layer (base + adapter)

class LoRALinear(nn.Module):
    """Linear layer with LoRA adapter.
    
    Computes: y = W @ x + B @ A @ x * scaling
    
    In vLLM, this is implemented differently — the LoRA computation
    is batched across requests using SGMV kernels.
    """
    def __init__(self, base_layer: nn.Linear, rank: int = 16, alpha: float = 16.0):
        super().__init__()
        self.base_layer = base_layer
        self.lora = LoRALayer(
            base_layer.in_features,
            base_layer.out_features,
            rank=rank,
            alpha=alpha
        )
        # Freeze base weights
        for p in self.base_layer.parameters():
            p.requires_grad = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_layer(x)
        lora_out = self.lora(x)
        return base_out + lora_out
    
    def merge(self) -> nn.Linear:
        """Merge LoRA weights into base layer (for deployment without overhead)."""
        merged = nn.Linear(
            self.base_layer.in_features,
            self.base_layer.out_features,
            bias=self.base_layer.bias is not None
        )
        delta = self.lora.get_merged_delta()
        merged.weight.data = self.base_layer.weight.data + delta
        if self.base_layer.bias is not None:
            merged.bias.data = self.base_layer.bias.data.clone()
        return merged


# Test LoRA-augmented linear
base = nn.Linear(512, 512, bias=False)
lora_linear = LoRALinear(base, rank=8, alpha=8.0)

x = torch.randn(2, 512)
y_base = base(x)
y_lora = lora_linear(x)
y_diff = (y_lora - y_base).norm()

print(f"Base output norm: {y_base.norm().item():.4f}")
print(f"LoRA output norm: {y_lora.norm().item():.4f}")
print(f"Difference norm:  {y_diff.item():.4f}")
print(f"\nTrainable params: {sum(p.numel() for p in lora_linear.parameters() if p.requires_grad):,}")
print(f"Frozen params:    {sum(p.numel() for p in lora_linear.parameters() if not p.requires_grad):,}")

# Test merging
merged = lora_linear.merge()
y_merged = merged(x)
merge_error = (y_merged - y_lora).norm().item()
print(f"\nMerge verification error: {merge_error:.8f} (should be ~0)")

---
## Part 2: The Multi-LoRA Challenge

The key challenge: **serve many LoRA adapters concurrently** using a single base model.

### Why is this hard?
1. Different requests may use different adapters
2. In a continuous batching system, a single batch may contain requests for 10+ different adapters
3. We can't merge all adapters into the base model (they conflict)
4. Naive implementation: loop over each adapter separately → kills throughput

### vLLM's Solution
- **SGMV (Segmented Grouped Matrix-Vector multiplication)**: A custom CUDA kernel that batches LoRA computations across different adapters in a single kernel launch
- **LoRA Manager**: Manages loading, unloading, and GPU memory for adapters
- **Adapter-tagged requests**: Each request carries its adapter ID through the pipeline

In [ ]:
# Simulate the multi-LoRA batching problem

class MultiLoRABatch:
    """Demonstrates how a batch contains requests for different adapters.
    
    In continuous batching, a single forward pass processes tokens for
    multiple requests, each potentially using a different LoRA adapter.
    """
    def __init__(self):
        self.requests = []
    
    def add_request(self, request_id: str, adapter_name: Optional[str], num_tokens: int):
        self.requests.append({
            'request_id': request_id,
            'adapter': adapter_name,
            'num_tokens': num_tokens
        })
    
    def get_adapter_groups(self) -> Dict[str, List[int]]:
        """Group token indices by adapter."""
        groups = {}
        offset = 0
        for req in self.requests:
            adapter = req['adapter'] or '__base__'
            if adapter not in groups:
                groups[adapter] = []
            groups[adapter].extend(range(offset, offset + req['num_tokens']))
            offset += req['num_tokens']
        return groups


# Simulate a realistic batch
batch = MultiLoRABatch()
batch.add_request("req-1", "lora-coding", 50)
batch.add_request("req-2", "lora-medical", 30)
batch.add_request("req-3", None, 20)            # base model, no adapter
batch.add_request("req-4", "lora-coding", 45)
batch.add_request("req-5", "lora-legal", 35)
batch.add_request("req-6", "lora-medical", 25)
batch.add_request("req-7", "lora-creative", 40)
batch.add_request("req-8", None, 15)            # base model

groups = batch.get_adapter_groups()
total_tokens = sum(req['num_tokens'] for req in batch.requests)

print(f"=== Multi-LoRA Batch ===")
print(f"Total requests: {len(batch.requests)}")
print(f"Total tokens: {total_tokens}")
print(f"Unique adapters: {len(groups)}")
print(f"\nAdapter groups:")
for adapter, indices in groups.items():
    print(f"  {adapter:.<25s} {len(indices):>3d} tokens (indices: {indices[:4]}...)")

In [ ]:
# Compare naive vs batched LoRA computation

def naive_multi_lora(
    x: torch.Tensor,
    base_weight: torch.Tensor,
    adapters: Dict[str, Tuple[torch.Tensor, torch.Tensor]],
    adapter_assignments: Dict[str, List[int]],
    scaling: float = 1.0
) -> torch.Tensor:
    """Naive implementation: loop over each adapter separately.
    
    This is what you'd get without SGMV. Each adapter requires a 
    separate matrix multiplication.
    """
    output = x @ base_weight.T  # Base computation for all tokens
    
    for adapter_name, indices in adapter_assignments.items():
        if adapter_name == '__base__':
            continue
        if adapter_name not in adapters:
            continue
        
        A, B = adapters[adapter_name]
        # Gather tokens for this adapter
        x_subset = x[indices]
        # Compute LoRA: B @ A @ x * scaling
        lora_out = (x_subset @ A.T @ B.T) * scaling
        # Scatter back
        output[indices] += lora_out
    
    return output


def sgmv_multi_lora(
    x: torch.Tensor,
    base_weight: torch.Tensor,
    stacked_A: torch.Tensor,  # (num_adapters, rank, d_in)
    stacked_B: torch.Tensor,  # (num_adapters, d_out, rank)
    adapter_indices: torch.Tensor,  # (total_tokens,) — which adapter per token
    scaling: float = 1.0
) -> torch.Tensor:
    """Simulated SGMV: all adapters computed in a single batched operation.
    
    In vLLM, this uses a custom CUDA kernel (SGMV/BGMV).
    Here we simulate the batched logic.
    """
    output = x @ base_weight.T  # Base computation
    
    # Gather A and B matrices for each token's adapter
    # This is where the SGMV kernel shines — it does this efficiently on GPU
    A_per_token = stacked_A[adapter_indices]  # (total_tokens, rank, d_in)
    B_per_token = stacked_B[adapter_indices]  # (total_tokens, d_out, rank)
    
    # Batched LoRA computation
    # For each token: B[i] @ A[i] @ x[i]
    x_expanded = x.unsqueeze(-1)  # (total_tokens, d_in, 1)
    intermediate = torch.bmm(A_per_token, x_expanded)  # (total_tokens, rank, 1)
    lora_out = torch.bmm(B_per_token, intermediate).squeeze(-1)  # (total_tokens, d_out)
    
    output += lora_out * scaling
    return output


# Benchmark
d_in, d_out, rank = 512, 512, 16
num_adapters = 4
total_tokens = 260  # From our batch above

base_weight = torch.randn(d_out, d_in)
x = torch.randn(total_tokens, d_in)

# Create adapters
adapters = {}
adapter_names = ['lora-coding', 'lora-medical', 'lora-legal', 'lora-creative']
for name in adapter_names:
    A = torch.randn(rank, d_in) * 0.01
    B = torch.zeros(d_out, rank)
    adapters[name] = (A, B)

# Time naive approach
start = time.time()
for _ in range(100):
    out_naive = naive_multi_lora(x, base_weight, adapters, groups)
naive_time = (time.time() - start) / 100

# Setup SGMV
stacked_A = torch.stack([adapters[n][0] for n in adapter_names])  # (4, r, d_in)
stacked_B = torch.stack([adapters[n][1] for n in adapter_names])  # (4, d_out, r)

# Map: adapter name -> index
name_to_idx = {n: i for i, n in enumerate(adapter_names)}
adapter_indices = torch.zeros(total_tokens, dtype=torch.long)
for name, idxs in groups.items():
    if name in name_to_idx:
        for i in idxs:
            adapter_indices[i] = name_to_idx[name]

start = time.time()
for _ in range(100):
    out_sgmv = sgmv_multi_lora(x, base_weight, stacked_A, stacked_B, adapter_indices)
sgmv_time = (time.time() - start) / 100

print(f"Naive multi-LoRA:  {naive_time*1000:.3f} ms per batch")
print(f"SGMV multi-LoRA:   {sgmv_time*1000:.3f} ms per batch")
print(f"Speedup:           {naive_time/sgmv_time:.2f}x")
print(f"\n(On CPU the difference is modest; on GPU with custom CUDA kernels, SGMV is much faster)")

---
## Part 3: vLLM LoRA Manager — Adapter Lifecycle

vLLM's `LoRAManager` handles:
1. **Loading** adapters from disk (HuggingFace format)
2. **Storing** adapter weights on GPU
3. **Evicting** adapters when GPU memory is full (LRU policy)
4. **Mapping** request adapter IDs to loaded adapter slots

Key source: `vllm/lora/lora_manager.py`

In [ ]:
@dataclass
class LoRAAdapter:
    """Represents a loaded LoRA adapter."""
    adapter_id: str
    rank: int
    alpha: float
    target_modules: List[str]  # e.g., ['q_proj', 'v_proj', 'k_proj', 'o_proj']
    weights: Dict[str, Tuple[torch.Tensor, torch.Tensor]]  # module_name -> (A, B)
    loaded_at: float = 0.0
    last_used: float = 0.0
    request_count: int = 0
    
    @property
    def memory_bytes(self) -> int:
        total = 0
        for name, (A, B) in self.weights.items():
            total += A.numel() * A.element_size()
            total += B.numel() * B.element_size()
        return total
    
    @property
    def memory_mb(self) -> float:
        return self.memory_bytes / (1024 * 1024)


class LoRAManager:
    """Manages multiple LoRA adapters with memory budget.
    
    Key responsibilities:
    1. Load adapters (simulated here)
    2. Track loaded adapters and their memory usage
    3. Evict least-recently-used adapters when memory is full
    4. Map adapter IDs to GPU slots
    """
    def __init__(
        self,
        max_lora_memory_mb: float = 1024.0,  # 1GB budget for LoRA weights
        max_num_adapters: int = 16,
        base_model_dims: Dict[str, Tuple[int, int]] = None
    ):
        self.max_memory_mb = max_lora_memory_mb
        self.max_num_adapters = max_num_adapters
        self.base_model_dims = base_model_dims or {
            'q_proj': (4096, 4096),
            'k_proj': (4096, 1024),
            'v_proj': (4096, 1024),
            'o_proj': (4096, 4096),
        }
        
        # Active adapters (ordered dict for LRU)
        self.loaded_adapters: OrderedDict[str, LoRAAdapter] = OrderedDict()
        self.current_memory_mb = 0.0
        
        # Statistics
        self.stats = {
            'loads': 0, 'evictions': 0, 'cache_hits': 0, 'total_requests': 0
        }
    
    def _create_adapter_weights(
        self, rank: int, target_modules: List[str]
    ) -> Dict[str, Tuple[torch.Tensor, torch.Tensor]]:
        """Simulate loading adapter weights (normally loaded from disk)."""
        weights = {}
        for module_name in target_modules:
            if module_name in self.base_model_dims:
                d_in, d_out = self.base_model_dims[module_name]
            else:
                d_in, d_out = 4096, 4096
            
            A = torch.randn(rank, d_in, dtype=torch.float16) * 0.01
            B = torch.zeros(d_out, rank, dtype=torch.float16)
            weights[module_name] = (A, B)
        
        return weights
    
    def _evict_lru(self) -> Optional[str]:
        """Evict the least recently used adapter."""
        if not self.loaded_adapters:
            return None
        
        # LRU: first item in OrderedDict
        lru_id = next(iter(self.loaded_adapters))
        adapter = self.loaded_adapters.pop(lru_id)
        self.current_memory_mb -= adapter.memory_mb
        self.stats['evictions'] += 1
        return lru_id
    
    def load_adapter(
        self,
        adapter_id: str,
        rank: int = 16,
        alpha: float = 16.0,
        target_modules: List[str] = None
    ) -> LoRAAdapter:
        """Load an adapter (evicting LRU if necessary)."""
        target_modules = target_modules or ['q_proj', 'v_proj']
        
        # Check if already loaded
        if adapter_id in self.loaded_adapters:
            # Move to end (most recently used)
            self.loaded_adapters.move_to_end(adapter_id)
            adapter = self.loaded_adapters[adapter_id]
            adapter.last_used = time.time()
            self.stats['cache_hits'] += 1
            return adapter
        
        # Create weights
        weights = self._create_adapter_weights(rank, target_modules)
        adapter = LoRAAdapter(
            adapter_id=adapter_id,
            rank=rank,
            alpha=alpha,
            target_modules=target_modules,
            weights=weights,
            loaded_at=time.time(),
            last_used=time.time()
        )
        
        # Evict if necessary
        while (self.current_memory_mb + adapter.memory_mb > self.max_memory_mb 
               or len(self.loaded_adapters) >= self.max_num_adapters):
            evicted = self._evict_lru()
            if evicted is None:
                raise RuntimeError("Cannot load adapter: no space even after eviction")
        
        self.loaded_adapters[adapter_id] = adapter
        self.current_memory_mb += adapter.memory_mb
        self.stats['loads'] += 1
        
        return adapter
    
    def get_adapter(self, adapter_id: str) -> Optional[LoRAAdapter]:
        """Get a loaded adapter (updates LRU)."""
        if adapter_id in self.loaded_adapters:
            self.loaded_adapters.move_to_end(adapter_id)
            adapter = self.loaded_adapters[adapter_id]
            adapter.last_used = time.time()
            adapter.request_count += 1
            self.stats['total_requests'] += 1
            return adapter
        return None
    
    def status(self) -> dict:
        return {
            'loaded_adapters': len(self.loaded_adapters),
            'max_adapters': self.max_num_adapters,
            'memory_used_mb': round(self.current_memory_mb, 2),
            'memory_budget_mb': self.max_memory_mb,
            'memory_utilization': round(self.current_memory_mb / self.max_memory_mb * 100, 1),
            'adapters': [
                {
                    'id': a.adapter_id,
                    'rank': a.rank,
                    'memory_mb': round(a.memory_mb, 2),
                    'request_count': a.request_count
                }
                for a in self.loaded_adapters.values()
            ],
            'stats': self.stats
        }


# Demo
manager = LoRAManager(max_lora_memory_mb=5.0, max_num_adapters=4)

# Load several adapters
adapter_configs = [
    ("coding-assistant", 16),
    ("medical-qa", 32),
    ("legal-summarizer", 8),
    ("creative-writer", 16),
]

for name, rank in adapter_configs:
    adapter = manager.load_adapter(name, rank=rank, target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'])
    print(f"Loaded '{name}' (rank={rank}): {adapter.memory_mb:.2f} MB")

print(f"\nStatus: {json.dumps(manager.status(), indent=2)}")

In [ ]:
# Demo: LRU eviction under memory pressure
print("=== LRU Eviction Demo ===")
print(f"Current adapters: {list(manager.loaded_adapters.keys())}")
print(f"Memory: {manager.current_memory_mb:.2f} / {manager.max_memory_mb:.2f} MB")

# Use some adapters to change LRU order
manager.get_adapter("creative-writer")  # Move to most recent
manager.get_adapter("medical-qa")        # Move to most recent

# Load a new adapter — should evict the LRU (coding-assistant)
print(f"\nLoading new adapter 'translation-ja'...")
adapter = manager.load_adapter("translation-ja", rank=16, target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'])
print(f"Current adapters: {list(manager.loaded_adapters.keys())}")
print(f"Memory: {manager.current_memory_mb:.2f} / {manager.max_memory_mb:.2f} MB")
print(f"Stats: {manager.stats}")

# Load another — should evict legal-summarizer
print(f"\nLoading 'sentiment-analysis'...")
manager.load_adapter("sentiment-analysis", rank=8, target_modules=['q_proj', 'v_proj'])
print(f"Current adapters: {list(manager.loaded_adapters.keys())}")
print(f"Stats: {manager.stats}")

---
## Part 4: SGMV Kernel — Batched LoRA Computation

The **SGMV (Segmented Grouped Matrix-Vector)** kernel is the core innovation that makes multi-LoRA serving efficient. It handles the case where tokens in a batch use different LoRA adapters.

### How SGMV Works

```
Input:  x = [x1, x2, x3, x4, x5]   — tokens from different requests
Labels: l = [ 0,  1,  0,  2,  1]   — adapter index per token

Adapters:
  A[0], B[0] — adapter 0
  A[1], B[1] — adapter 1
  A[2], B[2] — adapter 2

SGMV computes in one kernel launch:
  y[0] = B[0] @ A[0] @ x[0]
  y[1] = B[1] @ A[1] @ x[1]
  y[2] = B[0] @ A[0] @ x[2]
  y[3] = B[2] @ A[2] @ x[3]
  y[4] = B[1] @ A[1] @ x[4]
```

In [ ]:
class SGMVSimulator:
    """Simulates the SGMV kernel for multi-LoRA computation.
    
    In vLLM, this is implemented as a CUDA kernel in:
    vllm/lora/punica_wrapper/punica_gpu.py
    
    The key insight: instead of launching N separate kernels for N adapters,
    SGMV groups all operations and executes them in a single kernel.
    """
    def __init__(self, adapters: Dict[int, Tuple[torch.Tensor, torch.Tensor]]):
        self.adapters = adapters
        # Stack for batched access
        self.stacked_A = torch.stack([adapters[i][0] for i in sorted(adapters.keys())])
        self.stacked_B = torch.stack([adapters[i][1] for i in sorted(adapters.keys())])
    
    def forward(
        self,
        x: torch.Tensor,               # (total_tokens, d_in)
        adapter_indices: torch.Tensor,  # (total_tokens,) — adapter ID per token
        scaling: float = 1.0
    ) -> torch.Tensor:
        """Compute batched LoRA for all tokens.
        
        This simulates SGMV in pure PyTorch.
        The real CUDA kernel is much more efficient on GPU.
        """
        # Gather A and B for each token
        A = self.stacked_A[adapter_indices]  # (N, r, d_in)
        B = self.stacked_B[adapter_indices]  # (N, d_out, r)
        
        # Batched computation: y[i] = B[i] @ A[i] @ x[i]
        x_3d = x.unsqueeze(-1)              # (N, d_in, 1)
        intermediate = torch.bmm(A, x_3d)    # (N, r, 1)
        output = torch.bmm(B, intermediate)  # (N, d_out, 1)
        
        return output.squeeze(-1) * scaling


# Demo: SGMV computation
d_in, d_out, rank = 256, 256, 8
num_adapters = 3
total_tokens = 20

# Create adapters
adapters = {
    i: (torch.randn(rank, d_in) * 0.01, torch.randn(d_out, rank) * 0.01)
    for i in range(num_adapters)
}

sgmv = SGMVSimulator(adapters)

# Create input and adapter assignments
x = torch.randn(total_tokens, d_in)
adapter_indices = torch.randint(0, num_adapters, (total_tokens,))

# Compute
output = sgmv.forward(x, adapter_indices)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Adapter assignments: {adapter_indices.tolist()}")

# Verify against naive computation
output_naive = torch.zeros_like(output)
for i in range(total_tokens):
    aid = adapter_indices[i].item()
    A, B = adapters[aid]
    output_naive[i] = (B @ A @ x[i])

error = (output - output_naive).norm().item()
print(f"\nVerification error vs naive: {error:.8f} (should be ~0)")

In [ ]:
# Benchmark: scaling behavior with number of adapters
import time

d_in, d_out, rank = 512, 512, 16
total_tokens = 256
num_iters = 50

adapter_counts = [1, 2, 4, 8, 16, 32]
naive_times = []
sgmv_times = []

for n_adapters in adapter_counts:
    adapters = {
        i: (torch.randn(rank, d_in) * 0.01, torch.randn(d_out, rank) * 0.01)
        for i in range(n_adapters)
    }
    x = torch.randn(total_tokens, d_in)
    adapter_indices = torch.randint(0, n_adapters, (total_tokens,))
    
    # Naive timing
    start = time.time()
    for _ in range(num_iters):
        out = torch.zeros(total_tokens, d_out)
        for aid in range(n_adapters):
            mask = adapter_indices == aid
            if mask.any():
                A, B = adapters[aid]
                x_sub = x[mask]
                out[mask] = x_sub @ A.T @ B.T
    naive_times.append((time.time() - start) / num_iters * 1000)
    
    # SGMV timing
    sgmv = SGMVSimulator(adapters)
    start = time.time()
    for _ in range(num_iters):
        out = sgmv.forward(x, adapter_indices)
    sgmv_times.append((time.time() - start) / num_iters * 1000)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = range(len(adapter_counts))
width = 0.35
ax.bar([p - width/2 for p in x_pos], naive_times, width, label='Naive (loop)', color='salmon')
ax.bar([p + width/2 for p in x_pos], sgmv_times, width, label='SGMV (batched)', color='steelblue')
ax.set_xticks(x_pos)
ax.set_xticklabels(adapter_counts)
ax.set_xlabel('Number of Active Adapters')
ax.set_ylabel('Time (ms) per batch')
ax.set_title(f'Multi-LoRA Performance: Naive vs SGMV\n({total_tokens} tokens, d={d_in}, rank={rank})')
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'Adapters':<10} {'Naive (ms)':<12} {'SGMV (ms)':<12} {'Speedup':<10}")
for n, nt, st in zip(adapter_counts, naive_times, sgmv_times):
    print(f"{n:<10} {nt:<12.3f} {st:<12.3f} {nt/st:<10.2f}x")

---
## Part 5: Dynamic LoRA Loading and Unloading

In production, adapters come and go dynamically. vLLM supports:
- Loading adapters on-the-fly via the API
- Automatic eviction when memory is full
- The `--enable-lora` flag and `--lora-modules` configuration

In [ ]:
class DynamicLoRAServer:
    """Simulates a vLLM server with dynamic LoRA loading.
    
    Launch vLLM with LoRA support:
      python -m vllm.entrypoints.openai.api_server \
        --model meta-llama/Llama-3-8B-Instruct \
        --enable-lora \
        --lora-modules coding=./adapters/coding medical=./adapters/medical \
        --max-loras 8 \
        --max-lora-rank 64
    """
    
    def __init__(self, max_adapters: int = 8, memory_budget_mb: float = 100.0):
        self.manager = LoRAManager(
            max_lora_memory_mb=memory_budget_mb,
            max_num_adapters=max_adapters
        )
        self.request_log = []
    
    def handle_request(self, prompt: str, adapter_id: Optional[str] = None) -> dict:
        """Handle an inference request, loading adapter if needed."""
        start = time.time()
        
        adapter = None
        if adapter_id:
            adapter = self.manager.get_adapter(adapter_id)
            if adapter is None:
                # Need to load it
                adapter = self.manager.load_adapter(
                    adapter_id, rank=16,
                    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj']
                )
        
        # Simulate inference
        time.sleep(0.001)  # Tiny delay
        latency = (time.time() - start) * 1000
        
        result = {
            'prompt': prompt[:50],
            'adapter': adapter_id or 'base',
            'latency_ms': round(latency, 2),
            'generated': f"[Response using {adapter_id or 'base model'}]"
        }
        self.request_log.append(result)
        return result


# Simulate realistic traffic
server = DynamicLoRAServer(max_adapters=4, memory_budget_mb=5.0)

traffic = [
    ("Write a Python sort function", "coding-v1"),
    ("What are the symptoms of flu?", "medical-v2"),
    ("Write a Python web scraper", "coding-v1"),
    ("Summarize this legal contract", "legal-v1"),
    ("Translate to Japanese", "translate-ja"),
    ("Write a poem about the sea", "creative-v1"),    # Will evict LRU
    ("Debug this Python error", "coding-v1"),         # Cache hit or reload
    ("Explain this medical test", "medical-v2"),
    ("What is the weather?", None),                    # Base model
    ("Write a haiku", "creative-v1"),
]

print("=== Dynamic LoRA Traffic Simulation ===")
for prompt, adapter in traffic:
    result = server.handle_request(prompt, adapter)
    loaded = list(server.manager.loaded_adapters.keys())
    print(f"  [{result['adapter']:.<18s}] latency={result['latency_ms']:.1f}ms | loaded={loaded}")

print(f"\nFinal stats: {server.manager.stats}")

In [ ]:
# Visualize adapter loading/eviction timeline
fig, ax = plt.subplots(figsize=(14, 6))

# Track which adapters are loaded at each step
server2 = DynamicLoRAServer(max_adapters=4, memory_budget_mb=5.0)

all_adapters = set()
timeline = []
for i, (prompt, adapter) in enumerate(traffic):
    server2.handle_request(prompt, adapter)
    loaded = set(server2.manager.loaded_adapters.keys())
    all_adapters.update(loaded)
    timeline.append(loaded)

adapter_list = sorted(all_adapters)
for j, adapter_name in enumerate(adapter_list):
    for i, loaded in enumerate(timeline):
        if adapter_name in loaded:
            ax.barh(j, 1, left=i, height=0.6, color=plt.cm.Set3(j % 12), edgecolor='black', linewidth=0.5)

ax.set_yticks(range(len(adapter_list)))
ax.set_yticklabels(adapter_list)
ax.set_xlabel('Request Number')
ax.set_title('Adapter Loading Timeline (max 4 concurrent)')
ax.set_xticks(range(len(traffic)))

# Annotate which adapter each request used
for i, (_, adapter) in enumerate(traffic):
    ax.text(i + 0.5, -0.8, adapter or 'base', rotation=45, fontsize=7, ha='right')

plt.tight_layout()
plt.show()

---
## Part 6: Merged vs Separate LoRA Computation

Two approaches to serving a LoRA model:

1. **Merged**: Merge LoRA weights into base weights: `W' = W + B*A*scaling`. Fast inference but can only serve one adapter.
2. **Separate**: Keep LoRA weights separate and add during forward pass. Slightly slower but enables multi-LoRA.

In [ ]:
# Compare merged vs separate LoRA performance

d_in, d_out = 1024, 1024
batch_sizes = [1, 4, 16, 64, 256]
rank = 16

base = nn.Linear(d_in, d_out, bias=False)
lora_linear = LoRALinear(base, rank=rank)
merged_linear = lora_linear.merge()

separate_times = []
merged_times = []

for bs in batch_sizes:
    x = torch.randn(bs, d_in)
    
    # Separate
    start = time.time()
    for _ in range(200):
        y = lora_linear(x)
    separate_times.append((time.time() - start) / 200 * 1000)
    
    # Merged
    start = time.time()
    for _ in range(200):
        y = merged_linear(x)
    merged_times.append((time.time() - start) / 200 * 1000)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = range(len(batch_sizes))
width = 0.35
ax.bar([p - width/2 for p in x_pos], separate_times, width, label='Separate (base + LoRA)', color='salmon')
ax.bar([p + width/2 for p in x_pos], merged_times, width, label='Merged (W + ΔW)', color='steelblue')
ax.set_xticks(x_pos)
ax.set_xticklabels(batch_sizes)
ax.set_xlabel('Batch Size')
ax.set_ylabel('Time (ms)')
ax.set_title(f'Merged vs Separate LoRA (d={d_in}, rank={rank})')
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'Batch':<8} {'Separate (ms)':<15} {'Merged (ms)':<15} {'Overhead':<10}")
for bs, st, mt in zip(batch_sizes, separate_times, merged_times):
    overhead = (st - mt) / mt * 100
    print(f"{bs:<8} {st:<15.3f} {mt:<15.3f} {overhead:<10.1f}%")

print("\nKey insight: Merged is always faster for single-adapter serving.")
print("Separate is needed for multi-adapter serving (different requests use different adapters).")

---
## Part 7: LoRA Target Modules and Rank Selection

Which modules to apply LoRA to, and what rank to use, are important design decisions.

In [ ]:
# Analysis: which modules benefit most from LoRA
# Based on empirical findings from the LoRA paper and follow-ups

module_configs = {
    'q_proj only': ['q_proj'],
    'q+v (original LoRA paper)': ['q_proj', 'v_proj'],
    'q+k+v+o (QLoRA default)': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    'All linear (full LoRA)': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
}

# Llama-3 8B dimensions
module_dims = {
    'q_proj': (4096, 4096),
    'k_proj': (4096, 1024),   # GQA: smaller K
    'v_proj': (4096, 1024),   # GQA: smaller V
    'o_proj': (4096, 4096),
    'gate_proj': (4096, 14336),
    'up_proj': (4096, 14336),
    'down_proj': (14336, 4096),
}
num_layers = 32

print(f"=== LoRA Parameter Analysis for Llama-3 8B ({num_layers} layers) ===")
print(f"Base model parameters: ~8B\n")

print(f"{'Config':<35} {'Rank 8':<15} {'Rank 16':<15} {'Rank 32':<15} {'Rank 64':<15}")
print("-" * 95)

for config_name, modules in module_configs.items():
    params_by_rank = []
    for rank in [8, 16, 32, 64]:
        total = 0
        for mod in modules:
            d_in, d_out = module_dims[mod]
            total += rank * (d_in + d_out) * num_layers
        params_by_rank.append(total)
    
    print(f"{config_name:<35} ", end='')
    for p in params_by_rank:
        print(f"{p/1e6:<15.1f}M", end='')
    print()

print(f"\nMemory per adapter (fp16, in MB):")
print(f"{'Config':<35} {'Rank 16':<15}")
print("-" * 50)
for config_name, modules in module_configs.items():
    total_bytes = 0
    for mod in modules:
        d_in, d_out = module_dims[mod]
        total_bytes += 16 * (d_in + d_out) * num_layers * 2  # fp16
    print(f"{config_name:<35} {total_bytes/1024/1024:<15.1f}")

---
## Hands-On Assignments

### Assignment 1: Implement a Simplified LoRA Layer

**Task**: Implement a complete LoRA-augmented transformer attention module.

Requirements:
1. Create `LoRAAttention` that wraps Q, K, V, O projections with LoRA
2. Support configurable rank per projection (different ranks for Q vs V)
3. Implement `merge()` and `unmerge()` methods
4. Test that merged output matches separate computation
5. Count and report parameter statistics

In [ ]:
# === ASSIGNMENT 1: LoRA-Augmented Attention ===

class LoRAAttention(nn.Module):
    """
    TODO: Implement a self-attention module with LoRA on Q, K, V, O projections.
    
    Features:
    - Configurable rank per projection
    - merge() to fold LoRA into base weights
    - unmerge() to separate them again
    - Parameter counting (trainable vs frozen)
    """
    def __init__(
        self,
        d_model: int = 512,
        num_heads: int = 8,
        lora_config: Dict[str, int] = None  # e.g., {'q': 16, 'v': 16, 'k': 0, 'o': 0}
    ):
        super().__init__()
        # YOUR CODE HERE
        pass
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass
    
    def merge(self):
        # YOUR CODE HERE
        pass
    
    def unmerge(self):
        # YOUR CODE HERE
        pass

print("Assignment 1: Implement LoRAAttention with merge/unmerge support.")
print("Test: verify that merged output equals separate computation.")

### Assignment 2: Build a LoRA Adapter Manager with LRU Eviction

**Task**: Build a production-grade LoRA manager with advanced eviction policies.

Requirements:
1. Support multiple eviction policies: LRU, LFU (Least Frequently Used), and weighted (combines recency + frequency)
2. Implement adapter priority levels (some adapters should never be evicted)
3. Add warmup: pre-load a set of adapters at startup
4. Track detailed statistics per adapter
5. Implement a `simulate_traffic()` method that replays a request trace

In [ ]:
# === ASSIGNMENT 2: Advanced LoRA Manager ===

class AdvancedLoRAManager:
    """
    TODO: Implement an advanced LoRA manager.
    
    Features:
    - Multiple eviction policies (LRU, LFU, weighted)
    - Adapter priority levels (pinned adapters)
    - Warmup with pre-loaded adapters
    - Detailed per-adapter statistics
    - Traffic simulation and hit-rate analysis
    """
    def __init__(self, max_adapters: int, memory_budget_mb: float,
                 eviction_policy: str = 'lru'):
        # YOUR CODE HERE
        pass
    
    def load(self, adapter_id: str, rank: int = 16, priority: str = 'normal'):
        # YOUR CODE HERE
        pass
    
    def evict(self) -> Optional[str]:
        # YOUR CODE HERE
        pass
    
    def simulate_traffic(self, trace: List[str]) -> dict:
        # YOUR CODE HERE
        pass

print("Assignment 2: Build AdvancedLoRAManager with LRU/LFU eviction.")
print("Test with a simulated traffic trace and compare hit rates.")

### Assignment 3: Benchmark Multi-LoRA Throughput vs Single Model

**Task**: Build a comprehensive benchmark comparing multi-LoRA serving performance.

Requirements:
1. Measure throughput (tokens/sec) for:
   - Base model only (no LoRA)
   - Single LoRA adapter (merged)
   - Single LoRA adapter (separate)
   - Multi-LoRA with 2, 4, 8, 16 adapters
2. Vary batch size and measure the impact
3. Vary LoRA rank and measure the overhead
4. Generate a comprehensive report with plots

In [ ]:
# === ASSIGNMENT 3: Multi-LoRA Throughput Benchmark ===

def multi_lora_benchmark(
    d_model: int = 512,
    ranks: List[int] = [8, 16, 32],
    num_adapters_list: List[int] = [1, 2, 4, 8],
    batch_sizes: List[int] = [1, 16, 64, 256],
    num_iters: int = 100
) -> dict:
    """
    TODO: Implement a comprehensive multi-LoRA benchmark.
    
    Measure:
    - Throughput (tokens/sec)
    - Latency (ms per batch)
    - Overhead vs base model (%)
    
    Compare across:
    - Number of adapters
    - Batch sizes
    - LoRA ranks
    """
    # YOUR CODE HERE
    pass

print("Assignment 3: Implement multi_lora_benchmark() and generate performance plots.")
print("Compare throughput across different adapter counts, ranks, and batch sizes.")

---
## Summary

| Concept | Description | Key Parameter |
|---------|-------------|---------------|
| LoRA Decomposition | ΔW = B × A with rank r | `--lora-rank` |
| Multi-LoRA Serving | Multiple adapters on one base model | `--enable-lora` |
| SGMV Kernel | Batched LoRA computation | Internal |
| LoRA Manager | Adapter loading/eviction | `--max-loras` |
| Memory Budget | GPU memory for LoRA weights | `--max-lora-memory` |
| Target Modules | Which layers get LoRA | Adapter config |

### Key Takeaways
1. LoRA enables serving 100+ fine-tuned variants from a single base model
2. SGMV kernels make multi-LoRA nearly as fast as single-model inference
3. LRU eviction manages GPU memory when there are more adapters than can fit
4. Rank 16-32 covers most use cases; higher ranks give diminishing returns
5. Apply LoRA to Q+V projections at minimum; all linear layers for maximum quality

In [ ]:
print("=== End of Chapter 4.8: LoRA Serving ===")
print("\nvLLM LoRA commands:")
print("  Enable:   --enable-lora")
print("  Modules:  --lora-modules name1=path1 name2=path2")
print("  Max:      --max-loras 16")
print("  Rank:     --max-lora-rank 64")
print("\nAPI usage:")
print('  curl -X POST http://localhost:8000/v1/chat/completions \\')
print('    -d \'{"model": "coding-adapter", "messages": [...]}\'')